# Ranking: Feature Engineering

In [1]:
import sys
import os
%load_ext autoreload
%autoreload 2

sys.path.append(os.path.abspath("../src"))

## Spark + Data Initialisation
Initialising Spark and loading data saved as parquets from retrieval stage

In [2]:
from utils.spark_session import get_spark

spark = get_spark()
spark.sparkContext.setLogLevel("INFO")

In [3]:
# Reloading data from ALS retrieval stage:
# recommendations per user from ALS
# test data

als_candidates = spark.read.parquet('../data/retrieval/als_candidates.parquet')
test = spark.read.parquet('../data/retrieval/test_filtered.parquet')
train = spark.read.parquet('../data/retrieval/train.parquet')

## User Features

In [4]:
from pyspark.sql import functions as F
als_candidates.groupBy('userId').agg(
    F.count(F.col('rating')).alias('num_rat')
).orderBy('num_rat').show(10)

+------+-------+
|userId|num_rat|
+------+-------+
| 83090|     58|
|118205|     69|
| 34576|     90|
|    65|    100|
|   458|    100|
|   879|    100|
|   883|    100|
|  1223|    100|
|  1977|    100|
|  2096|    100|
+------+-------+
only showing top 10 rows


The following users have < 100 ratings due to duplicate removals:
| User | Rating |
| 83090|         58|
|118205|         69|
| 34576|         90|

### Build User Features

In [4]:
from data.stats import compute_global_std

global_std = compute_global_std(train)
global_std

1.0544861563196177

In [5]:
from features.user_features import build_user_features

user_feature = build_user_features(train, global_std, k_shrinkage=20)
user_feature.show(5)

+------+------------------+-----------------+------------------+---------------------+------------------------+------------------+
|userId|   user_avg_rating|user_rating_count|   user_rating_std|user_log_rating_count|days_since_last_activity|    user_bayes_std|
+------+------------------+-----------------+------------------+---------------------+------------------------+------------------+
|   148|3.6666666666666665|              105|1.2142318453899053|    4.663439094112067|                    4732|1.1886725351386591|
|   463|3.8636363636363638|               66|0.7822327734684298|    4.204692619390966|                    6879|0.8455475136663805|
|   471| 3.519406392694064|              438| 1.026073190102175|   6.0844994130751715|                    3582|1.0273139309850325|
|   496| 4.140740740740741|              135|0.7547161467292026|    4.912654885736052|                    5243|0.7933961479666756|
|   833|3.4864864864864864|               37|1.0171207079469644|   3.63758615972638

## Item Features

In [6]:
from data.stats import compute_global_mean

mu = compute_global_mean(train)
mu

3.538268676182596

In [7]:
# build out item features
from features.item_features import build_item_features

item_feature = build_item_features(train, mu, C=50)
item_feature.show(5)

+-------+------------------+-----------------+------------------+---------------------+
|movieId|   item_avg_rating|item_rating_count| item_bayesian_avg|item_log_rating_count|
+-------+------------------+-----------------+------------------+---------------------+
|   1580|3.5540448674371174|            30891|3.5540193734465313|   10.338252529689154|
|   3918|2.9090019569471623|             1022|2.9383520837771733|    6.930494765951626|
|   1591|2.6172638436482085|             4298|2.6278549755770766|    8.366137716496281|
|    471| 3.663809007053717|             9215|3.6631315093156105|    9.128696382935672|
|  44022| 3.334841628959276|             1768| 3.340436432238245|    7.478169694159785|
+-------+------------------+-----------------+------------------+---------------------+
only showing top 5 rows


## Biases

In [8]:
# Hyperparameters
bias_hparams = {
    'reg_param': 10,
    'tau': 3,
    'epsilon': 1e-6
}

### Item Biases

In [9]:
from features.biases import compute_item_bias

item_bias = compute_item_bias(item_feature, mu=mu, reg_param=bias_hparams['reg_param'])
item_bias.show(5)

+-------+--------------------+
|movieId|           item_bias|
+-------+--------------------+
|   1580|  0.0157710858562318|
|   3918| -0.6231691735064082|
|   1591| -0.9188669383084488|
|    471|  0.1254042437915856|
|  44022|-0.20228291309945426|
+-------+--------------------+
only showing top 5 rows


### User Biases

In [10]:
from features.biases import compute_user_bias

user_bias = compute_user_bias(train, user_feature, item_bias, mu=mu, reg_param=bias_hparams['reg_param'])
user_bias.show(5)

+------+--------------------+
|userId|           user_bias|
+------+--------------------+
|    62|  0.5537397407296536|
|   125| 0.09412485405020397|
|  1453|-0.12730375874570835|
|  1907|  0.1756637595038785|
|  2529|   0.661608128781244|
+------+--------------------+
only showing top 5 rows


## Residuals & Weighting

### Expected Rating

In [11]:
from features.biases import compute_expected_rating

expected_rating = compute_expected_rating(train, user_bias, item_bias, mu)
expected_rating.show(5)

+-------+------+------+--------------------+-------------------+------------------+
|movieId|userId|rating|           user_bias|          item_bias|   expected_rating|
+-------+------+------+--------------------+-------------------+------------------+
|    924|     1|   3.5|-0.03672005592564839|0.42766021774718316| 3.929208838004131|
|    919|     1|   3.5|-0.03672005592564839|0.44574436509396315|3.9472929853509107|
|   2683|     1|   3.5|-0.03672005592564839|-0.3268373072780361|3.1747113129789115|
|   1584|     1|   3.5|-0.03672005592564839|0.11003201992666994|3.6115806401836177|
|   1079|     1|   4.0|-0.03672005592564839|0.31855615373045276|3.8201047739874006|
+-------+------+------+--------------------+-------------------+------------------+
only showing top 5 rows


## Weightings

In [12]:
from features.biases import compute_user_weights

weights = compute_user_weights(
    expected_rating, user_feature, tau=bias_hparams['tau'],epsilon=bias_hparams['epsilon'])
weights.show(5)

+------+-------+--------------------+
|userId|movieId|              weight|
+------+-------+--------------------+
|     1|    924|-0.29595845298235973|
|     1|    919|  -0.307641754894343|
|     1|   2683|  0.2271830067147221|
|     1|   1584|-0.07914630528418612|
|     1|   1079|  0.1271782042762094|
+------+-------+--------------------+
only showing top 5 rows


## Tag Features

### PCA dimension reduction

In [13]:
import configs.settings as cfg
from features.tag_features import build_genome_pca_features
# load csv as dataframe
genomes = spark.read.csv("../data/raw/genome_scores.csv", schema=cfg.GENOMIC_SCHEMA, header=True)

# conduct pca and scaling on genome features
genome_scalar_model, genome_pca_df, genome_pca_model =  (
    build_genome_pca_features(genomes, k=45)
)

In [6]:
# Saving genome pca dataframe
genome_pca_df.write.mode('overwrite').parquet('../data/features/genome_pca_45.parquet')

### Normalise Tags

In [14]:
from data.preprocessing import normaliser
item_tag_norm = normaliser(genome_pca_df, input_col='pca_features', output_col='item_tag_norm')
item_tag_norm.show(5)

+-------+--------------------+
|movieId|       item_tag_norm|
+-------+--------------------+
|    496|[0.07888750859851...|
|    148|[-0.0275135375140...|
|    463|[-0.6914273308849...|
|    471|[0.40777863452039...|
|    833|[-0.6868075439068...|
+-------+--------------------+
only showing top 5 rows


## User Vector

In [15]:
from features.user_vectors import build_user_tags

user_tags = build_user_tags(weights, genome_pca_df)
user_tags.show(5)

+------+--------------------+
|userId|            user_tag|
+------+--------------------+
|     1|[-1.0873354972383...|
|     6|[-0.0428293335896...|
|    12|[-0.5091748335730...|
|    13|[0.41907874989096...|
|    16|[-0.8635531020686...|
+------+--------------------+
only showing top 5 rows


In [16]:
from data.preprocessing import normaliser

user_tag_norm = normaliser(user_tags, input_col='user_tag', output_col='user_tag_norm')
user_tag_norm.show(5)

+------+--------------------+
|userId|       user_tag_norm|
+------+--------------------+
|     1|[-0.7879113244960...|
|     6|[-0.0499329459155...|
|    12|[-0.4393815476588...|
|    13|[0.42979596005678...|
|    16|[-0.7996641673526...|
+------+--------------------+
only showing top 5 rows


## Cosine Similarity
Compare user average tag (genome scores) to films

In [41]:
from ranking.tag_similarity import compute_tag_similarity

similarity_score = compute_tag_similarity(user_tag_norm, item_tag_norm, als_candidates)

similarity_score.orderBy('userId').show(150)

+------+-------+----------+--------------------+
|userId|movieId| als_score|          similarity|
+------+-------+----------+--------------------+
|     1|   2571|0.90139693|-0.19600672973806982|
|     1|   3793| 0.8977427| 0.24953806833403397|
|     1|   1197|0.89429057| 0.04830728135420092|
|     1|   1206|  0.893153| -0.6097246448837004|
|     1|   3703|  0.871913|-0.11447985152064939|
|     1|   1220| 0.8628025|-0.32009553690905806|
|     1|   1274| 0.8529236|-0.40818368873038186|
|     1|   1210|0.83239865| 0.04392144314488101|
|     1|   1275|0.80829763| 0.26646572323189965|
|     1|   5618| 0.8021501| -0.2174659540494162|
|     1|    968| 0.8007482|-0.23258785356499118|
|     1|   2987|0.80019194|-0.05257869253544...|
|     1|   2019|0.79848516| -0.5744995722185154|
|     1|   2115| 0.7925773| 0.25899534056435464|
|     1|   8874| 0.7913283| -0.3065155975837094|
|     1|   1748| 0.7831828| -0.2670125117420579|
|     1|   1213|0.78060234| -0.6767655782537263|
|     1|   1288| 0.7

# Final Feature Dataset

In [42]:
from ranking.ranking_df import create_ranking_df

ranking = create_ranking_df(similarity_score, user_feature, item_feature, user_bias, item_bias)
ranking.orderBy('userId').show(200)

+-------+------+----------+--------------------+------------------+-------------------+---------------------+------------------------+------------------+------------------+------------------+---------------------+--------------------+--------------------+
|movieId|userId| als_score|          similarity|   user_avg_rating|    user_rating_std|user_log_rating_count|days_since_last_activity|    user_bayes_std|   item_avg_rating| item_bayesian_avg|item_log_rating_count|           user_bias|           item_bias|
+-------+------+----------+--------------------+------------------+-------------------+---------------------+------------------------+------------------+------------------+------------------+---------------------+--------------------+--------------------+
|   2571|     1|0.90139693|-0.19600672973806982|3.7785714285714285|0.38530296671089637|    4.948759890378168|                    3650|0.4689508654119865| 4.197637486947442| 4.196921062058523|   10.735722535246286|-0.0367200559256483

In [43]:
# Saving final ranking data as parquet
ranking.write.mode('overwrite').parquet('../data/ranking/ranking_candidates.parquet')

In [44]:
ranking.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- userId: integer (nullable = true)
 |-- als_score: float (nullable = true)
 |-- similarity: double (nullable = true)
 |-- user_avg_rating: double (nullable = true)
 |-- user_rating_std: double (nullable = false)
 |-- user_log_rating_count: double (nullable = true)
 |-- days_since_last_activity: integer (nullable = true)
 |-- user_bayes_std: double (nullable = true)
 |-- item_avg_rating: double (nullable = true)
 |-- item_bayesian_avg: double (nullable = true)
 |-- item_log_rating_count: double (nullable = true)
 |-- user_bias: double (nullable = true)
 |-- item_bias: double (nullable = true)

